In [5]:
import os
import re
from dotenv import load_dotenv
from litellm import completion

load_dotenv()


def extract_python_code(text: str) -> str:
    match = re.search(r"```python\s*\n(.*?)```", text, re.DOTALL)
    if not match:
        return ""
    return match.group(1).strip()
    
schema = """
    ser (string), 
    status (string), 
    score (float), 
    quantity (int), 
    priority (string), 
    event_date (string), 
    extra (float), 
    flag (bool),
    misc (string),
    label (bool)
"""

code_template = """
import skrub
import pandas as pd
from skrub import DropUninformative, TableVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier

df = pd.read_csv('experiments/globalprogramsynthesis/data.csv')
all_data = skrub.var("all_data", df)

labels = all_data['label'].skb.mark_as_y()
data = all_data.drop(columns=['label']).skb.mark_as_X()

dropper = DropUninformative(drop_if_constant=True, drop_if_unique=True, drop_null_fraction=0.9)
cleaned_data = data.skb.apply(dropper)

#TODO: Add ten cool extra features derived from existing columns

encoded_data = with_features.skb.apply(TableVectorizer())

predictions = encoded_data.skb.apply(HistGradientBoostingClassifier(), y=labels)
"""

model = "gemini/gemini-2.5-pro"
model_name = "gemini25pro"

for repetition in range(10):
    print(f'Iteration {repetition}...')
    response = completion(
        model=model,
        messages=[{"role": "user", "content": f"""
            Complete the following ML pipeline code with suitable operations from common libraries pandas, torch, etc.  
            The input data has the following schema: {schema}
            ONLY INSERT CODE for the #TODOs, don't change the rest of the script. 
            COMPUTE NEW COLUMNS on pandas df with the .assign function only, do the same for date conversions via pd.to_datetime
            RESPOND WITH THE FULL CODE.  
            -----
            
            {code_template}
        """}],
        temperature=0.5,
    )
    
    code = extract_python_code(response.choices[0].message.content)
    with open(f'experiments/globalprogramsynthesis/{model_name}_intermediate_{repetition}.py', "w") as f:
        f.write(code)

Iteration 0...
Iteration 1...
Iteration 2...
Iteration 3...
Iteration 4...
Iteration 5...
Iteration 6...
Iteration 7...
Iteration 8...
Iteration 9...
